# Verified local sandbox evaluation

This notebook uses actual Linux Docker execution, real LangGraph/Pydantic/SQLite, and deterministic offline reasoning. The optimizer fault-injection examples alter the grader only; learner execution is never mocked. No OpenAI inference is claimed.

In [1]:
from pathlib import Path
import os, sys, tempfile, json
workspace=Path.cwd()
if not (workspace/'src').exists():workspace=workspace.parent
sys.path.insert(0,str(workspace))
run_dir=tempfile.TemporaryDirectory(prefix='sprints-verified-notebook-')
os.environ.update(SPRINTS_LLM='offline',SPRINTS_EMBEDDINGS='hashing',SPRINTS_LIVE_AUTHORIZED='0',SPRINTS_DB_PATH=str(Path(run_dir.name)/'learners.db'),SPRINTS_CHROMA_DIR=str(Path(run_dir.name)/'chroma'),ANONYMIZED_TELEMETRY='False',LANGSMITH_TRACING='false',LANGCHAIN_TRACING_V2='false')
from src.auth import MockIdentityProvider
from src.service import OrchestratorService
from src.main import BUGGY,FIXED
from src.tools.code_repl import sandbox_capabilities
from src.tools.db_queries import build_database,get_learner_profile
capability=sandbox_capabilities()
print(json.dumps(capability,indent=2))
assert capability['available'], 'Verified Linux sandbox is required; no fixture substitution'
build_database()
identity=MockIdentityProvider()
session=identity.issue_mock_session('learner-1005')

{
  "available": true,
  "backend": "linux-docker",
  "image": "sha256:fc0ef3811e9d81b95a5912d21ada33d5fe432712724e8a33f9a7e32e0a5a0a2d",
  "controls": "no-network,non-root,read-only,no-mounts,seccomp,limits"
}


## Actual container grading
Both submissions go through the unchanged five-second isolated executor. A timeout or infrastructure failure remains a failure; no invented grade substitutes for it.

In [2]:
app=OrchestratorService(identity,warm=False)
actual=[]
for submission in (BUGGY,FIXED):
    result=app.ask(session,{'text':submission})
    actual.append(result)
    print(json.dumps(result,indent=2))
assert all(r['status']=='graded' for r in actual)
assert actual[0]['assessment']['score']<100
assert actual[1]['assessment']['score']==100

{
  "status": "graded",
  "route": "assessor",
  "final_answer": "Score: 60/100 - FAIL (pass mark 70)\n- Passed 3/6 tests (correctness 35/70).\n- Test failed for input ([80, None, 90],): expected 85.0, got 56.67.\n- Test failed for input ([],): expected 0.0, got ZeroDivisionError: division by zero.\n- Readability: add a docstring explaining what the code does (-5).",
  "turn_count": 1,
  "assessment": {
    "score": 60,
    "feedback_points": [
      "Passed 3/6 tests (correctness 35/70).",
      "Test failed for input ([80, None, 90],): expected 85.0, got 56.67.",
      "Test failed for input ([],): expected 0.0, got ZeroDivisionError: division by zero.",
      "Readability: add a docstring explaining what the code does (-5)."
    ],
    "pass_status": "fail"
  },
  "refinement_count": 0,
  "grader_calls": 1,
  "needs_human_review": false,
  "retrieval_attempts": 0
}


{
  "status": "graded",
  "route": "assessor",
  "final_answer": "Score: 100/100 - PASS (pass mark 70)\n- Passed 6/6 tests (correctness 70/70).\n- All tests passed and the code is clean and readable. Excellent work.",
  "turn_count": 2,
  "assessment": {
    "score": 100,
    "feedback_points": [
      "Passed 6/6 tests (correctness 70/70).",
      "All tests passed and the code is clean and readable. Excellent work."
    ],
    "pass_status": "pass"
  },
  "refinement_count": 0,
  "grader_calls": 1,
  "needs_human_review": false,
  "retrieval_attempts": 0
}


## Bounded optimizer with an injected faulty grader
These two grader fixtures intentionally inflate the first or every draft. The graph, container execution, and deterministic judge use real runtime evidence. The judge is not an LLM. The first case must correct one draft; the second must stop at three refinements and withhold a final grade.

In [3]:
from tests.test_evaluator_optimizer import LearnsFromJudge,Inflater
optimized=[]
for reasoner in (LearnsFromJudge(),Inflater()):
    result=OrchestratorService(identity,reasoner=reasoner,warm=False).ask(session,{'text':BUGGY})
    optimized.append(result)
    print('GRADER_FAULT_INJECTION_REAL_CONTAINER',type(reasoner).__name__,json.dumps(result,indent=2))
assert optimized[0]['refinement_count']==1
assert optimized[0]['status']=='graded'
assert optimized[1]['refinement_count']==3
assert optimized[1]['grader_calls']==4
assert optimized[1]['status']=='graded_needs_human_review'

GRADER_FAULT_INJECTION_REAL_CONTAINER LearnsFromJudge {
  "status": "graded",
  "route": "assessor",
  "final_answer": "Score: 60/100 - FAIL (pass mark 70)\n- Passed 3/6 tests (correctness 35/70).\n- Test failed for input ([80, None, 90],): expected 85.0, got 56.67.\n- Test failed for input ([],): expected 0.0, got ZeroDivisionError: division by zero.\n- Readability: add a docstring explaining what the code does (-5).",
  "turn_count": 1,
  "assessment": {
    "score": 60,
    "feedback_points": [
      "Passed 3/6 tests (correctness 35/70).",
      "Test failed for input ([80, None, 90],): expected 85.0, got 56.67.",
      "Test failed for input ([],): expected 0.0, got ZeroDivisionError: division by zero.",
      "Readability: add a docstring explaining what the code does (-5)."
    ],
    "pass_status": "fail"
  },
  "refinement_count": 1,
  "grader_calls": 2,
  "needs_human_review": false,
  "retrieval_attempts": 0
}


GRADER_FAULT_INJECTION_REAL_CONTAINER Inflater {
  "status": "graded_needs_human_review",
  "route": "assessor",
  "final_answer": "Provisional score: 100/100 - NOT FINAL (pending review by a Sprints grader)\n- Passed 6/6 tests. Fixture inflated score.\nNote: the automatic judge did not approve this grade after 3 refinements, so a Sprints grader will double-check this result before it counts.",
  "turn_count": 1,
  "assessment": {
    "score": 100,
    "feedback_points": [
      "Passed 6/6 tests. Fixture inflated score."
    ],
    "pass_status": "pass"
  },
  "refinement_count": 3,
  "grader_calls": 4,
  "needs_human_review": true,
  "retrieval_attempts": 0
}


In [4]:
with identity.bind(session):
    record=json.loads(get_learner_profile.invoke({'learner_id':'SPR-1005'}))
assert record['status']=='ok'
print(json.dumps({k:record[k] for k in ('learner_id','completed_modules','python_assignment_score','target_salary_egp','preferred_employers')},indent=2))
print('No paid API requests; business waiting-time improvement is unmeasured.')

{
  "learner_id": "SPR-1005",
  "completed_modules": 10,
  "python_assignment_score": 92,
  "target_salary_egp": 35000,
  "preferred_employers": [
    "e& Egypt",
    "Fawry"
  ]
}
No paid API requests; business waiting-time improvement is unmeasured.


## Scope
Passing this notebook establishes local execution and integration for these cases. Python restriction escapes, shared-kernel container vulnerabilities, same-interpreter verdict integrity, production authentication, live LLM decision quality and business waiting-time reduction need separate evaluation. Original media remain unavailable.